In [36]:
import json
import pandas as pd
import os

train_path = '/home/work/hocheol_dir/workspace/data/data_in_use/0522_userId/train.json'
test_path = '/home/work/hocheol_dir/workspace/data/data_in_use/0522_userId/test.json'
val_path = '/home/work/hocheol_dir/workspace/data/data_in_use/0522_userId/val.json'

with open(train_path, 'r') as f:
    train_data = json.load(f)
with open(test_path, 'r') as f:
    test_data = json.load(f)
with open(val_path, 'r') as f:
    val_data = json.load(f)

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)
val_df = pd.DataFrame(val_data)

In [37]:
import json
from sklearn.model_selection import StratifiedKFold

def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def save_json(data, path):
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

def apply_stratified_kfold(json_path, n_splits=5):
    data = load_json(json_path)
    labels = [d["label"] for d in data]
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    datasets = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(data, labels)):
        val_data = [data[i] for i in val_idx]
        datasets.append(val_data)
    
    return datasets

# ================== 사용 예시 ==================
train_datasets = apply_stratified_kfold(
    json_path=train_path,
    n_splits=5
)

val_datasets = apply_stratified_kfold(
    json_path=val_path,
    n_splits=5
)

test_datasets = apply_stratified_kfold(
    json_path=test_path,
    n_splits=5
)

In [38]:
len(train_df), len(val_df), len(test_df)

(78846, 9327, 9476)

In [ ]:
sum([len(d) for d in train_datasets]), sum([len(d) for d in val_datasets]), sum([len(d) for d in test_datasets])

(78846, 9327, 9476)

In [45]:
train_df2 = pd.DataFrame([d for train_dataset in train_datasets for d in train_dataset])
val_df2 = pd.DataFrame([d for val_dataset in val_datasets for d in val_dataset])
test_df2 = pd.DataFrame([d for test_dataset in test_datasets for d in test_dataset])

In [61]:
import os

fold_dir = '/home/work/hocheol_dir/workspace/data/data_in_use/0522_split_5Fold'

for i, dataset in enumerate(train_datasets):
    fold_path = os.path.join(fold_dir, f'train_fold_{i}.json')
    os.makedirs(os.path.dirname(fold_path), exist_ok=True)
    save_json(dataset, fold_path)

for i, dataset in enumerate(val_datasets):
    fold_path = os.path.join(fold_dir, f'val_fold_{i}.json')
    os.makedirs(os.path.dirname(fold_path), exist_ok=True)
    save_json(dataset, fold_path)
    
for i, dataset in enumerate(test_datasets):
    fold_path = os.path.join(fold_dir, f'test_fold_{i}.json')
    os.makedirs(os.path.dirname(fold_path), exist_ok=True)
    save_json(dataset, fold_path)

In [62]:
for i in range(5):
    os.makedirs(os.path.join(fold_dir, f'fold_{i}'), exist_ok=True)